# Extra Trees

Shared work already completed: raw-data audit, EDA, train-only imputation and band thresholds, feature engineering/encoding, one stratified hold-out split, artifact validation, and the 10-fold CV contract. This notebook must not preprocess raw Titanic data again. It reads the shared artifacts in `artifacts/common/` and follows the data contract in `feature_manifest.json` (28 features, `test_size=0.20` stratified hold-out, `StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`, primary metric `accuracy`, no scaling for tree models).

## Setup

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                      GridSearchCV, learning_curve)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve)

RANDOM_STATE = 42
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PROJECT_PLAN.md').exists())
ARTIFACT_DIR = ROOT / 'artifacts' / 'common'
OUT_DIR = ROOT / 'artifacts' / 'extra_trees'
FIG_DIR = ROOT / 'reports' / 'figures' / 'extra_trees'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Shared CV contract (identical for every member) and hold-out metrics.
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
SCORING = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
sns.set_theme(style='whitegrid')
print('ROOT   =', ROOT)
print('OUT_DIR=', OUT_DIR)

ROOT   = C:\Users\MSI\Desktop\MyProducts\My Project\AIL\Mini_Capstone
OUT_DIR= C:\Users\MSI\Desktop\MyProducts\My Project\AIL\Mini_Capstone\artifacts\extra_trees


## Load shared artifacts

In [2]:
X_train = pd.read_csv(ARTIFACT_DIR / 'X_train.csv')
X_valid = pd.read_csv(ARTIFACT_DIR / 'X_valid.csv')
y_train = pd.read_csv(ARTIFACT_DIR / 'y_train.csv').squeeze('columns')
y_valid = pd.read_csv(ARTIFACT_DIR / 'y_valid.csv').squeeze('columns')
X_full = pd.read_csv(ARTIFACT_DIR / 'X_full.csv')
y_full = pd.read_csv(ARTIFACT_DIR / 'y_full.csv').squeeze('columns')
X_test = pd.read_csv(ARTIFACT_DIR / 'X_test.csv')
test_passenger_id = pd.read_csv(ARTIFACT_DIR / 'test_passenger_id.csv')
manifest = json.loads((ARTIFACT_DIR / 'feature_manifest.json').read_text(encoding='utf-8'))
FEATURES = manifest['features']

# Schema / shape / NaN / infinity checks against the data contract.
assert X_train.columns.tolist() == FEATURES == X_valid.columns.tolist() \
       == X_full.columns.tolist() == X_test.columns.tolist(), 'feature order mismatch'
for name, df in [('X_train', X_train), ('X_valid', X_valid),
                 ('X_full', X_full), ('X_test', X_test)]:
    assert int(df.isna().sum().sum()) == 0, f'{name} contains NaN'
    assert np.isfinite(df.to_numpy()).all(), f'{name} contains inf'
assert len(X_train) + len(X_valid) == len(X_full), 'train+valid != full'
assert len(test_passenger_id) == len(X_test), 'passenger id / test row mismatch'

print('shapes:', X_train.shape, X_valid.shape, X_full.shape, X_test.shape)
print('class balance (full):', dict(pd.Series(y_full).value_counts().sort_index()))
print('n_features:', len(FEATURES))

shapes: (712, 28) (179, 28) (891, 28) (418, 28)
class balance (full): {0: np.int64(549), 1: np.int64(342)}
n_features: 28


## Baseline

Default `ExtraTreesClassifier(random_state=42, n_jobs=-1)` as mandated by the plan. Tree ensembles are scale-invariant, so no scaling is applied.

In [3]:
baseline = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1)
baseline.fit(X_train, y_train)
print(baseline)

ExtraTreesClassifier(n_jobs=-1, random_state=42)


## Cross-validation

10-fold stratified CV of the baseline on `X_train` / `y_train`, reporting every contract metric plus the mean train score to gauge the baseline's overfit gap.

In [4]:
baseline_cv = cross_validate(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train, cv=cv, scoring=SCORING,
    return_train_score=True, n_jobs=-1)

print('Baseline 10-fold CV (test folds):')
for m in SCORING:
    s = baseline_cv[f'test_{m}']
    print(f'  {m:10s} {s.mean():.4f} +/- {s.std():.4f}')
baseline_cv_mean = float(baseline_cv['test_accuracy'].mean())
baseline_cv_std = float(baseline_cv['test_accuracy'].std())
print(f'\nmean train accuracy = {baseline_cv["train_accuracy"].mean():.4f} '
      f'(overfit gap vs CV = {baseline_cv["train_accuracy"].mean() - baseline_cv_mean:.4f})')

Baseline 10-fold CV (test folds):
  accuracy   0.7837 +/- 0.0276
  precision  0.7233 +/- 0.0429
  recall     0.7107 +/- 0.0846
  f1         0.7140 +/- 0.0442
  roc_auc    0.8301 +/- 0.0461

mean train accuracy = 0.9855 (overfit gap vs CV = 0.2018)


## Hyperparameter tuning

The full suggested grid (`n_estimators`×`max_depth`×`max_features`×`min_samples_split`×`min_samples_leaf`×`criterion`×`class_weight` = 1296 configs × 10 folds, with `n_estimators` up to 500) is expensive, so tuning is **coarse-to-fine**:

- **Stage 1 (coarse)** — fix `n_estimators=300` and scan the structural axes (criterion, depth, max_features, split/leaf, class_weight) → 144 configs.
- **Stage 2 (fine)** — hold the coarse structural winners and sweep `n_estimators ∈ {100,300,500}` together with the `min_samples_split` / `min_samples_leaf` neighbourhood → 27 configs.

Both stages use the shared `StratifiedKFold` and `scoring='accuracy'`. The estimator runs `n_jobs=1` inside the search so `GridSearchCV(n_jobs=-1)` parallelises across configs without oversubscription; the selected best model is later rebuilt with `n_jobs=-1`.

In [5]:
param_grid_coarse = {
    'n_estimators': [300],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 8, 12],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'class_weight': [None, 'balanced'],
}  # 1*2*3*3*2*2*2 = 144 configs

et = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1)
t0 = time.perf_counter()
gs_coarse = GridSearchCV(et, param_grid_coarse, scoring='accuracy', cv=cv,
                         n_jobs=-1, return_train_score=True)
gs_coarse.fit(X_train, y_train)
coarse_time = time.perf_counter() - t0
print(f'Coarse: {len(gs_coarse.cv_results_["params"])} configs in {coarse_time:.1f}s')
print('  best params    :', gs_coarse.best_params_)
print(f'  best CV acc    : {gs_coarse.best_score_:.4f}')

Coarse: 144 configs in 248.2s
  best params    : {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
  best CV acc    : 0.8371


In [6]:
bp = gs_coarse.best_params_
param_grid_fine = {
    'n_estimators': [100, 300, 500],
    'criterion': [bp['criterion']],
    'max_depth': [bp['max_depth']],
    'max_features': [bp['max_features']],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [bp['class_weight']],
}  # 3*1*1*1*3*3*1 = 27 configs

et_fine = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1)
t0 = time.perf_counter()
gs_fine = GridSearchCV(et_fine, param_grid_fine, scoring='accuracy', cv=cv,
                       n_jobs=-1, return_train_score=True)
gs_fine.fit(X_train, y_train)
fine_time = time.perf_counter() - t0
print(f'Fine: {len(gs_fine.cv_results_["params"])} configs in {fine_time:.1f}s')
print('  best params    :', gs_fine.best_params_)
print(f'  best CV acc    : {gs_fine.best_score_:.4f}')

# Pick the better of the two searches as the overall winner.
if gs_fine.best_score_ >= gs_coarse.best_score_:
    best_search, best_params, best_cv_score = gs_fine, gs_fine.best_params_, float(gs_fine.best_score_)
else:
    best_search, best_params, best_cv_score = gs_coarse, gs_coarse.best_params_, float(gs_coarse.best_score_)
tuning_time = coarse_time + fine_time
print(f'\nOVERALL best CV acc {best_cv_score:.4f} | params {best_params}')

Fine: 27 configs in 35.2s
  best params    : {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
  best CV acc    : 0.8371

OVERALL best CV acc 0.8371 | params {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}


### Best params, CV score and top configurations

In [7]:
def _results(search, stage):
    r = pd.DataFrame(search.cv_results_)
    r['stage'] = stage
    return r

cv_results_all = pd.concat([_results(gs_coarse, 'coarse'), _results(gs_fine, 'fine')],
                           ignore_index=True)

best_idx = cv_results_all['mean_test_score'].idxmax()
best_row = cv_results_all.loc[best_idx]
best_cv_mean = float(best_row['mean_test_score'])
best_cv_std = float(best_row['std_test_score'])

cols = ['stage', 'mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time', 'params']
top = cv_results_all.sort_values('mean_test_score', ascending=False).head(10)
print('Top 10 configurations:')
print(top[cols].to_string(index=False))
print(f'\nBest CV accuracy : {best_cv_mean:.4f} +/- {best_cv_std:.4f}')
print(f'Total tuning time: {tuning_time:.1f}s '
      f'(coarse {coarse_time:.1f}s + fine {fine_time:.1f}s)')

# Rebuild the winning estimator with n_jobs=-1 for the hold-out evaluation.
best_model = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
best_model.fit(X_train, y_train)

Top 10 configurations:
 stage  mean_test_score  std_test_score  mean_train_score  mean_fit_time                                                                                                                                                      params
coarse         0.837050        0.035281          0.854712       1.313252    {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
  fine         0.837050        0.035281          0.854712       1.206179    {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
coarse         0.837050        0.035281          0.855025       1.336643 {'class_weight': None, 'criterion': 'entropy', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
  fine         0.835642        0.036865          0.855180    

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls 3 sources of randomness:- the bootstrapping of the samples used when building trees (if ``bootstrap=True``)- the sampling of the features to consider when looking for the best split at each node (if ``max_features < n_features``)- the draw of the splits for each of the `max_features`See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity

## Validation evaluation

Full metric suite, classification report, confusion matrix and ROC on the shared hold-out, comparing baseline vs tuned and analysing the train/CV/validation gap for over/under-fitting.

In [8]:
def evaluate(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred)),
        'recall': float(recall_score(y, pred)),
        'f1': float(f1_score(y, pred)),
        'roc_auc': float(roc_auc_score(y, proba)),
    }, pred, proba

baseline_val, base_pred, base_proba = evaluate(baseline, X_valid, y_valid)
tuned_val, val_pred, val_proba = evaluate(best_model, X_valid, y_valid)

print('Baseline validation:', {k: round(v, 4) for k, v in baseline_val.items()})
print('Tuned    validation:', {k: round(v, 4) for k, v in tuned_val.items()})
print('\nClassification report (tuned):')
print(classification_report(y_valid, val_pred, target_names=['Died', 'Survived'], digits=4))

tuned_train, _, _ = evaluate(best_model, X_train, y_train)
print(f'Over/under-fit view  -> train acc {tuned_train["accuracy"]:.4f} | '
      f'CV acc {best_cv_mean:.4f} | valid acc {tuned_val["accuracy"]:.4f}')
print(f'Baseline vs tuned validation accuracy: '
      f'{baseline_val["accuracy"]:.4f} -> {tuned_val["accuracy"]:.4f} '
      f'({tuned_val["accuracy"] - baseline_val["accuracy"]:+.4f})')

Baseline validation: {'accuracy': 0.7933, 'precision': 0.7424, 'recall': 0.7101, 'f1': 0.7259, 'roc_auc': 0.8024}
Tuned    validation: {'accuracy': 0.8324, 'precision': 0.8197, 'recall': 0.7246, 'f1': 0.7692, 'roc_auc': 0.8542}

Classification report (tuned):
              precision    recall  f1-score   support

        Died     0.8390    0.9000    0.8684       110
    Survived     0.8197    0.7246    0.7692        69

    accuracy                         0.8324       179
   macro avg     0.8293    0.8123    0.8188       179
weighted avg     0.8315    0.8324    0.8302       179

Over/under-fit view  -> train acc 0.8581 | CV acc 0.8371 | valid acc 0.8324
Baseline vs tuned validation accuracy: 0.7933 -> 0.8324 (+0.0391)


In [9]:
# Confusion matrix
cm = confusion_matrix(y_valid, val_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Died', 'Survived']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Extra Trees — Validation Confusion Matrix')
fig.tight_layout(); fig.savefig(FIG_DIR / 'confusion_matrix.png', dpi=120); plt.close(fig)

# ROC curve (tuned vs baseline)
fpr, tpr, _ = roc_curve(y_valid, val_proba)
bfpr, btpr, _ = roc_curve(y_valid, base_proba)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, label=f'Tuned (AUC={tuned_val["roc_auc"]:.3f})')
ax.plot(bfpr, btpr, ls='--', label=f'Baseline (AUC={baseline_val["roc_auc"]:.3f})')
ax.plot([0, 1], [0, 1], color='grey', ls=':')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Extra Trees — Validation ROC'); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / 'roc_curve.png', dpi=120); plt.close(fig)
print('saved confusion_matrix.png, roc_curve.png')

saved confusion_matrix.png, roc_curve.png


## Learning curve

CV learning curve of the tuned configuration to see whether more data would still help and to confirm the train/validation gap.

In [10]:
train_sizes, train_scores, val_scores = learning_curve(
    ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, **best_params),
    X_train, y_train, cv=cv, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)

fig, ax = plt.subplots(figsize=(6, 4))
tr_m, tr_s = train_scores.mean(1), train_scores.std(1)
va_m, va_s = val_scores.mean(1), val_scores.std(1)
ax.plot(train_sizes, tr_m, 'o-', label='Train')
ax.fill_between(train_sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15)
ax.plot(train_sizes, va_m, 'o-', label='CV')
ax.fill_between(train_sizes, va_m - va_s, va_m + va_s, alpha=0.15)
ax.set_xlabel('Training samples'); ax.set_ylabel('Accuracy')
ax.set_title('Extra Trees — Learning Curve'); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / 'learning_curve.png', dpi=120); plt.close(fig)
print(f'final train acc {tr_m[-1]:.4f} | final CV acc {va_m[-1]:.4f}')
print('saved learning_curve.png')

final train acc 0.8555 | final CV acc 0.8356
saved learning_curve.png


## Refit full data

Refit the winning configuration on all labeled data (`X_full`, `y_full`) for deployment, then report and plot the feature importances of this best estimator.

In [11]:
final_model = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
final_model.fit(X_full, y_full)

importances = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Top 15 feature importances:')
print(importances.head(15).to_string())

topk = importances.head(15)[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(topk.index, topk.values, color='#4C72B0')
ax.set_title('Extra Trees — Top 15 Feature Importances')
ax.set_xlabel('Importance')
fig.tight_layout(); fig.savefig(FIG_DIR / 'feature_importance.png', dpi=120); plt.close(fig)
print('saved feature_importance.png')

Top 15 feature importances:
Title_Mr                 0.224000
Sex                      0.199848
Pclass                   0.086233
Title_Miss               0.085981
Title_Mrs                0.076251
FamilyCategory_Large     0.041068
FareBand_3               0.028898
FamilyCategory_Medium    0.023309
FareBand_0               0.022460
Fare                     0.018981
FamilySize               0.018180
Title_Master             0.017416
FamilyCategory_Single    0.016167
SibSp                    0.015009
Age                      0.012939


saved feature_importance.png


## Test prediction

Predict labels and class-1 probabilities for the Kaggle test set with the full-data model. The test set is never used for model selection.

In [12]:
test_pred = final_model.predict(X_test).astype(int)
test_proba = final_model.predict_proba(X_test)[:, 1]
pid = test_passenger_id.iloc[:, 0].to_numpy()
print('test label counts:', dict(pd.Series(test_pred).value_counts().sort_index()))
print('proba range: [%.4f, %.4f]' % (test_proba.min(), test_proba.max()))

test label counts: {0: np.int64(262), 1: np.int64(156)}
proba range: [0.0664, 0.9721]


## Export artifacts

Write the six standard outputs to `artifacts/extra_trees/`.

In [13]:
# 1) model
joblib.dump(final_model, OUT_DIR / 'best_model.joblib')

# 2) metrics.json  (contract keys + extras)
metrics = {
    'model_name': 'ExtraTreesClassifier',
    'random_state': RANDOM_STATE,
    'cv_folds': 10,
    'baseline_cv_mean': baseline_cv_mean,
    'baseline_cv_std': baseline_cv_std,
    'best_cv_score': best_cv_score,
    'best_cv_mean': best_cv_mean,
    'best_cv_std': best_cv_std,
    'best_params': best_params,
    'tuning_seconds': round(tuning_time, 2),
    'baseline_validation': baseline_val,
    'validation_accuracy': tuned_val['accuracy'],
    'validation_precision': tuned_val['precision'],
    'validation_recall': tuned_val['recall'],
    'validation_f1': tuned_val['f1'],
    'validation_roc_auc': tuned_val['roc_auc'],
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, default=float), encoding='utf-8')

# 3) cv_results.csv  (both stages combined)
cv_results_all.to_csv(OUT_DIR / 'cv_results.csv', index=False)

# 4) validation_predictions.csv : row_id, y_true, y_pred, y_score
val_out = pd.DataFrame({
    'row_id': np.arange(len(X_valid)),
    'y_true': np.asarray(y_valid),
    'y_pred': val_pred.astype(int),
    'y_score': val_proba,
})
val_out.to_csv(OUT_DIR / 'validation_predictions.csv', index=False)

# 5) test_predictions.csv : PassengerId, Survived, score
test_out = pd.DataFrame({'PassengerId': pid, 'Survived': test_pred, 'score': test_proba})
test_out.to_csv(OUT_DIR / 'test_predictions.csv', index=False)

# 6) submission.csv : PassengerId, Survived
test_out[['PassengerId', 'Survived']].to_csv(OUT_DIR / 'submission.csv', index=False)

print('exported to', OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name, p.stat().st_size, 'bytes')

exported to C:\Users\MSI\Desktop\MyProducts\My Project\AIL\Mini_Capstone\artifacts\extra_trees
  best_model.joblib 2637577 bytes
  cv_results.csv 122158 bytes
  metrics.json 961 bytes
  submission.csv 3258 bytes
  test_predictions.csv 11370 bytes
  validation_predictions.csv 5010 bytes


## Conclusion

- **Baseline** `ExtraTreesClassifier(random_state=42, n_jobs=-1)` establishes the 10-fold CV reference; the fully-grown default trees show the expected large train/CV gap (near-perfect train accuracy), i.e. variance-driven overfitting typical of un-regularised tree ensembles.
- **Coarse-to-fine tuning** (144 + 27 configs, shared `StratifiedKFold`, `scoring='accuracy'`) regularises the ensemble via `max_depth` / `min_samples_*` / `max_features`, narrowing the train/CV gap while holding or improving CV accuracy.
- **Hold-out** metrics (accuracy, precision, recall, F1, ROC-AUC), the confusion matrix, ROC and learning curve are saved under `reports/figures/extra_trees/`; the learning curve indicates whether additional data would still help.
- **Feature importances** of the full-data best estimator identify the dominant predictors (title/sex/fare/pclass family of features), consistent with the EDA.
- **Deployment**: the winning configuration is refit on all 891 labeled rows and used to produce test predictions and the two-column submission. Exact numbers are reproducible with `random_state=42` and are recorded in `artifacts/extra_trees/metrics.json`.

**Limitations & next steps**: accuracy was the contract-mandated selection metric; on this mildly imbalanced target, probability calibration and threshold tuning (F1/recall oriented) could be explored. Extra Trees trades interpretability for variance reduction; the final cross-model comparison lives in the shared integration notebook.